<a href="https://colab.research.google.com/github/dora-lai/fyp-aspect-financial-news/blob/main/Task68_Document_Level_Comparison_on_200_Gold_Set.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from transformers import pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# 1. Load your 200 Gold Set headlines
df = pd.read_csv("dora_gold_annotations_completed.csv")

# 2. Run FinBERT
print("Running FinBERT on the Gold Set...")
finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert")
finbert_results = finbert(df['sentence'].tolist(), batch_size=32)
df['finbert_pred'] = [r['label'].lower() for r in finbert_results]

# 3. Run VADER (Lexicon)
print("Running VADER on the Gold Set...")
nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()
def vader_predict(text):
    score = sia.polarity_scores(text)['compound']
    if score >= 0.05: return 'positive'
    elif score <= -0.05: return 'negative'
    else: return 'neutral'

df['vader_pred'] = df['sentence'].apply(vader_predict)

# 4. Evaluate and create the Comparison Table
models = [
    ('VADER (Lexicon)', 'vader_pred'),
    ('FinBERT (Document-level)', 'finbert_pred')
]

results = []
for name, col in models:
    acc = accuracy_score(df['sentiment'], df[col])
    p, r, f1_macro, _ = precision_recall_fscore_support(df['sentiment'], df[col], average='macro', zero_division=0)
    _, _, f1_weighted, _ = precision_recall_fscore_support(df['sentiment'], df[col], average='weighted', zero_division=0)

    results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'Macro F1': round(f1_macro, 4),
        'Weighted F1': round(f1_weighted, 4)
    })

comparison_table = pd.DataFrame(results)
print("\n--- Task 68: Document-Level Comparison on 200 Gold Set ---")
print(comparison_table.to_string(index=False))

# 5. Save the table for GitHub
comparison_table.to_csv("gold_set_comparison.csv", index=False)
df.to_csv("gold_set_with_predictions.csv", index=False)

Running FinBERT on the Gold Set...


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Running VADER on the Gold Set...

--- Task 68: Document-Level Comparison on 200 Gold Set ---
                   Model  Accuracy  Macro F1  Weighted F1
         VADER (Lexicon)    0.4583    0.4497       0.4578
FinBERT (Document-level)    0.6019    0.6023       0.6085


In [ ]:
import pandas as pd
from sklearn.metrics import classification_report
gold = pd.read_csv("dora_gold_annotations_completed.csv")
gold['sentence'] = gold['sentence'].astype(str).str.strip()
gold['sentiment'] = gold['sentiment'].astype(str).str.strip().str.lower()
pred = {s: predict(s) for s in gold['sentence'].unique()}     # one prediction per headline
gold['pred'] = gold['sentence'].map(pred).astype(str).str.strip().str.lower()
print(len(gold), gold['sentiment'].value_counts().to_dict())
print(gold['pred'].unique())                                  # must be exactly 3 labels
print(classification_report(gold['sentiment'], gold['pred'], digits=4, zero_division=0))

NameError: name 'predict' is not defined